# Hybrid2 — Unsupervised Keyphrase Extraction
Implementation based on:
> *Hybrid Approach To Unsupervised Keyphrase Extraction*  
> Singh & Bolla, Procedia Computer Science 235 (2024), 1498–1511

## Pipeline

```
Raw text
    ↓ clean_text()           — lowercasing and removal of special characters
Cleaned text
    ↓       ↓       ↓
 TopicRank YAKE    RAKE     — independent extraction
    ↓       ↓       ↓
  Min–max normalization     — comparable scores across methods
            ↓
      Hybrid2 fusion        — sum of normalized scores by keyphrase
            ↓
  Final rescaling [0, 1]    — uniform output score
```

## Key implementation decisions

| Step | Decision | Rationale / reference |
|---|---|---|
| Preprocessing | Cleaned text without stemming | Paper, Table 5: F1 = 0 with stemming |
| Normalization | Independent min–max normalization | TopicRank and RAKE scores use different scales |
| N-grams | Up to trigrams | Paper, §3.2.1: longer n-grams degrade results |
| Final score | Rescaled to [0, 1] | Practical addition for easier comparison |

## 1. Dependencies and setup

This notebook uses **pke** (TopicRank), **YAKE**, **rake-nltk**, **spaCy**, and **NLTK**. It does not require a GPU.

Run the installation cell in a suitable Python environment before starting the pipeline. The notebook downloads English and French spaCy models and NLTK resources. Internet access is required for initial installation. Dependency versions have not been pinned or independently validated.

In [ ]:
!pip install yake rake-nltk nltk --quiet
!pip install git+https://github.com/boudinfl/pke.git --quiet
!python -m spacy download en_core_web_sm --quiet
!python -m spacy download fr_core_news_sm --quiet
!python -m nltk.downloader stopwords punkt punkt_tab --quiet

## 2. Imports

In [ ]:
import re
import os
import time
from typing import List, Tuple, Dict
import yake
import pke
import nltk
from rake_nltk import Rake
from nltk.corpus import stopwords as nltk_stopwords

## 3. Parameters

Edit `LANGUE` and `N_KEYWORDS` as needed.

| Parameter | Supported values | Description |
|---|---|---|
| `LANGUE` | `'en'`, `'fr'` | Corpus language |
| `N_KEYWORDS` | Positive integer | Number of keyphrases returned |

In [ ]:
LANGUE = 'en'  # 'en' or 'fr'
N_KEYWORDS = 15  # number of keyphrases to extract
print(f'Language: {LANGUE} | Keyphrases: {N_KEYWORDS}')

## 4. Input corpus

Choose **one** of the following options by setting `CORPUS_SOURCE` in the configuration cell. The example is selected by default and requires no external files.

- `example`: run the self-contained demonstration in the next cell.
- `file`: load a local UTF-8 `.txt` file via `CORPUS_PATH`. Each nonempty line represents one document (for example, `Title.. Abstract`). This also works in Colab if the file is available in its filesystem.

The research corpora are **not distributed** with this repository. The original experiments used external scientific-text corpora; see the project README.

In [ ]:
# Option A — self-contained illustrative corpus (new demonstration text)
EXAMPLE_CORPUS = [
    "Machine learning algorithms learn patterns from data to make predictions. Supervised learning uses labeled examples to train classification and regression models.",
    "Neural networks consist of layers of interconnected units. Deep learning uses multiple layers to learn representations for images, text, and other data.",
    "Natural language processing analyzes written language. Keyphrase extraction identifies important terms and expressions in documents.",
    "Computer vision methods analyze images. Image classification predicts a category, while image segmentation assigns labels to pixels.",
    "Researchers evaluate models on datasets and compare results. Data preprocessing, feature extraction, and validation affect the reliability of machine learning experiments."
]
print(f'Example corpus available: {len(EXAMPLE_CORPUS)} documents')

In [ ]:
# Option B — use a local .txt corpus instead of the example.
# Set CORPUS_SOURCE = 'file' and provide CORPUS_PATH before running this cell.
from pathlib import Path

CORPUS_SOURCE = 'example'  # 'example' or 'file'
CORPUS_PATH = Path('../data/ml_en_junior.txt')  # one document per nonempty line

if CORPUS_SOURCE == 'example':
    CORPUS = EXAMPLE_CORPUS.copy()
elif CORPUS_SOURCE == 'file':
    if not CORPUS_PATH.is_file():
        raise FileNotFoundError(f'Corpus not found: {CORPUS_PATH.resolve()}')
    with CORPUS_PATH.open('r', encoding='utf-8') as f:
        CORPUS = [line.strip() for line in f if line.strip()]
else:
    raise ValueError("CORPUS_SOURCE must be 'example' or 'file'")

if not CORPUS:
    raise ValueError('The selected corpus is empty.')
print(f'Loaded {len(CORPUS)} documents (source: {CORPUS_SOURCE}).')

## 5. Preprocessing

Following **§3.2 of the reference paper**, preprocessing is limited to cleaned text:

1. Convert text to lowercase.
2. Remove wiki-style markup (wiki links and templates, etc.).
3. Remove nonalphabetic characters.
4. Normalize whitespace.

> **Stemming is intentionally excluded:** Table 5 of the paper reports F1 = 0 for the models evaluated with stemming.

The same cleaned text is passed to the three methods. Each method internally handles tokenization, stopwords, and candidate selection.

In [ ]:
def clean_text(text: str) -> str:
    """
    Clean raw text following paper section 3.2:
    lowercase → remove wiki markup → remove non-alphabetic chars → normalize spaces.
    Stemming is explicitly excluded (paper Table 5: F1=0 with stemming).
    """
    text = text.lower()
    # Remove wiki markup: '''bold''', ''italic'', [[link|label]], {{template}}
    text = re.sub(r"'{2,3}", ' ', text)
    text = re.sub(r'\[\[([^|\]]+\|)?([^\]]+)\]\]', r'\2', text)
    text = re.sub(r'\{\{[^}]*\}\}', ' ', text)
    text = re.sub(r'\[https?://\S+\s*([^\]]*)\]', r'\1', text)
    # Keep only alphabetic characters and spaces (including accented letters)
    text = re.sub(r'[^a-zàâäéèêëîïôùûüç\s]', ' ', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Apply unified pre-processing to the entire corpus
cleaned_corpus = [clean_text(doc) for doc in CORPUS]

# Preview
print('=== Raw text (first 200 chars) ===')
print(CORPUS[0][:200])
print('\n=== Cleaned text (first 200 chars) ===')
print(cleaned_corpus[0][:200])

## 6. Min–max normalization

The methods produce scores on **different scales**:

| Method | Typical range | Interpretation |
|---|---|---|
| TopicRank | ~0.001–0.05 | Higher is better |
| YAKE | ~0.0001–0.02 | **Lower is better** |
| RAKE | ~5–20 | Higher is better |

Without normalization, RAKE would dominate score fusion. Min–max normalization maps each method **independently** to `[0, 1]`. YAKE scores are then inverted (`1 - normalized_score`) so that higher normalized scores always indicate more relevant keyphrases.

In [ ]:
# Apply min-max normalisation to a list of (keyphrase, score) tuples
# For YAKE (lower score = better keyphrase), scores are inverted after normalisation so that higher always means more relevant
# Args:
#   results          : list of (keyphrase, raw_score)
#   higher_is_better : set to False for YAKE
# Returns:
#   dict {keyphrase_lowercase: normalised_score in [0, 1]}
def min_max_normalise(
    results: List[Tuple[str, float]],
    higher_is_better: bool = True
) -> Dict[str, float]:
    if not results:
        return {}
    scores = [s for _, s in results]
    min_s, max_s = min(scores), max(scores)
    denom = max_s - min_s if max_s != min_s else 1.0
    out = {}
    for kp, score in results:
        norm = (score - min_s) / denom
        out[kp.lower().strip()] = (1.0 - norm) if not higher_is_better else norm
    return out

print('min_max_normalise() defined')

## 7. Method 1 — TopicRank

TopicRank is a **graph-based** method:

1. Extract noun-phrase candidates using spaCy POS tagging.
2. Group similar candidates into topics using Jaccard similarity.
3. Build a topic graph weighted by co-occurrence.
4. Rank topics with PageRank and select a representative candidate from each topic.

The `pke` library internally handles tokenization, stopwords, and candidate selection.

In [ ]:
# Run TopicRank via the pke library
# Input : cleaned text from clean_text()
# Score : higher is better (PageRank-based)
# pke requires a loaded spaCy model object — loading it outside the function and passing it as argument avoids reloading from disk on every call
# Tokenisation, stopword removal and candidate selection are handled internally by pke/spaCy
# Args:
#   cleaned_corpus : list of cleaned document strings
#   lang           : 'en' or 'fr'
#   n              : number of keyphrases to return
#   nlp            : pre-loaded spaCy model (loaded once outside to save time)
# Returns:
#   (list of (keyphrase, score), elapsed_time_seconds)
def run_topicrank(
    cleaned_corpus : List[str],
    lang           : str,
    n              : int,
    nlp            = None   # pre-loaded spaCy model — pass it to avoid reloading on every call
) -> Tuple[List[Tuple[str, float]], float]:
    import spacy
    full_text  = ' '.join(cleaned_corpus)

    # Load the spaCy model only if not provided
    if nlp is None:
        spacy_model_name = 'fr_core_news_sm' if lang == 'fr' else 'en_core_web_sm'
        nlp = spacy.load(spacy_model_name)

    t0 = time.time()
    extractor = pke.unsupervised.TopicRank()
    extractor.load_document(input=full_text, language=lang, spacy_model=nlp)
    extractor.candidate_selection()
    extractor.candidate_weighting()
    results = extractor.get_n_best(n=n)
    elapsed = time.time() - t0

    return results, elapsed


# Load the spaCy model once here — reused across all calls to run_topicrank()
import spacy
spacy_model_name = 'fr_core_news_sm' if LANGUE == 'fr' else 'en_core_web_sm'
nlp = spacy.load(spacy_model_name)
print(f'spaCy model loaded : {spacy_model_name}')

tr_results, tr_time = run_topicrank(cleaned_corpus, LANGUE, N_KEYWORDS * 2, nlp=nlp)
print(f'TopicRank completed in {tr_time:.2f}s — {len(tr_results)} keyphrases extracted')

## 8. Method 2 — YAKE

YAKE (*Yet Another Keyword Extractor*, Campos et al., 2018) is a **statistical** method combining lexical features such as casing, position, frequency, and dispersion.

> **Inverted scoring convention:** a **lower** YAKE score indicates a **more relevant** keyphrase. The normalization step accounts for this automatically (`higher_is_better=False`).

In [ ]:
# Run YAKE via the official yake library
# Input : cleaned text from clean_text()
# Score : LOWER is better — inversion is handled in min_max_normalise()
# Tokenisation, stopword removal and candidate selection are handled internally
# Args:
#   cleaned_corpus : list of cleaned document strings
#   lang           : 'en' or 'fr'
#   n              : number of keyphrases to return
# Returns:
#   (list of (keyphrase, score), elapsed_time_seconds)
def run_yake(
    cleaned_corpus: List[str],
    lang: str,
    n: int
) -> Tuple[List[Tuple[str, float]], float]:
    full_text  = ' '.join(cleaned_corpus)
    yake_lang  = 'fr' if lang == 'fr' else 'en'

    t0 = time.time()
    extractor = yake.KeywordExtractor(
        lan=yake_lang,
        n=3,         # max n-gram size: unigrams, bigrams, trigrams (paper section 3.2.1)
        dedupLim=0.7,
        top=n
    )
    results = extractor.extract_keywords(full_text)
    elapsed = time.time() - t0

    return results, elapsed


yake_results, yake_time = run_yake(cleaned_corpus, LANGUE, N_KEYWORDS * 2)
print(f'YAKE completed in {yake_time:.2f}s — {len(yake_results)} keyphrases extracted')

## 9. Method 3 — RAKE

RAKE (*Rapid Automatic Keyword Extraction*, Rose et al., 2010) is a **statistical** approach based on the word degree-to-frequency ratio.

Stopwords act as phrase boundaries. A candidate's score is the sum of `degree(word) / frequency(word)` over its constituent words.

Each document is processed **separately** to prevent cleaned text without punctuation from becoming one giant candidate. The `rake-nltk` library handles stopword removal internally.

In [ ]:
# Run RAKE via the rake-nltk library
# Input : cleaned text from clean_text()
# Each document is processed separately to avoid one giant candidate when the corpus has no punctuation
# Score : higher is better
# Stopword removal and candidate selection are handled internally by rake-nltk
# Args:
#   cleaned_corpus : list of cleaned document strings
#   lang           : 'en' or 'fr'
#   n              : number of keyphrases to return
# Returns:
#   (list of (keyphrase, score), elapsed_time_seconds)
def run_rake(
    cleaned_corpus: List[str],
    lang: str,
    n: int
) -> Tuple[List[Tuple[str, float]], float]:
    nltk_lang = 'french' if lang == 'fr' else 'english'
    try:
        sw = nltk_stopwords.words(nltk_lang)
    except LookupError:
        nltk.download('stopwords', quiet=True)
        sw = nltk_stopwords.words(nltk_lang)

    t0 = time.time()
    cand_scores: Dict[str, float] = {}

    for doc in cleaned_corpus:
        r = Rake(
            stopwords=sw,
            min_length=1,
            max_length=3   # max trigrams, following paper section 3.2.1
        )
        r.extract_keywords_from_text(doc)
        for score, kp in r.get_ranked_phrases_with_scores():
            kp_lower = kp.lower().strip()
            # Keep the highest score if the same candidate appears in multiple documents
            if kp_lower not in cand_scores or score > cand_scores[kp_lower]:
                cand_scores[kp_lower] = score

    elapsed = time.time() - t0
    results = sorted(cand_scores.items(), key=lambda x: x[1], reverse=True)[:n]
    return results, elapsed


rake_results, rake_time = run_rake(cleaned_corpus, LANGUE, N_KEYWORDS * 2)
print(f'RAKE completed in {rake_time:.2f}s — {len(rake_results)} keyphrases extracted')

## 10. Method 4 — Hybrid2

Implementation of **Algorithm 2** from the paper (§3.2.2.2), with an additional final normalization to obtain scores in `[0, 1]`.

```
Algorithm 2 — Hybrid2
1. Input: TopicRank, YAKE, and RAKE results; desired number N.
2. Independently normalize the three methods' scores.
3. Create a dictionary mapping keyphrases to cumulative scores.
4. For each candidate from each method, add its normalized score
   to the existing entry or insert a new entry.
5. Sort by descending cumulative score and select the top N.
6. Output the final ranking (additionally rescaled to [0, 1] here).
```

**Why normalization matters:** a phrase found only by RAKE would otherwise have an unfair advantage due to the different raw score scales. After normalization, each method contributes at most 1 to the fused score.

In [ ]:
# Hybrid2 fusion following Algorithm 2 of the paper (section 3.2.2.2)
# Steps:
#   1. Min-max normalise each method independently (YAKE inverted: lower=better → higher_is_better=False)
#   2. Fusion dict: for each keyphrase, sum normalised scores across all methods that contain it
#   3. Sort by descending fused score
#   4. Select top-n keyphrases
#   5. Final re-normalisation to [0, 1] for uniform output scores
# Args:
#   topicrank_results : list of (keyphrase, score) from TopicRank
#   yake_results      : list of (keyphrase, score) from YAKE
#   rake_results      : list of (keyphrase, score) from RAKE
#   n                 : number of keyphrases to return
# Returns:
#   (list of (keyphrase, score in [0,1]), elapsed_time_seconds)
def run_hybrid2(
    topicrank_results : List[Tuple[str, float]],
    yake_results      : List[Tuple[str, float]],
    rake_results      : List[Tuple[str, float]],
    n                 : int
) -> Tuple[List[Tuple[str, float]], float]:
    t0 = time.time()

    # Step 1: independent min-max normalisation per method
    norm_tr   = min_max_normalise(topicrank_results, higher_is_better=True)
    norm_yake = min_max_normalise(yake_results,      higher_is_better=False)  # YAKE: lower = better
    norm_rake = min_max_normalise(rake_results,      higher_is_better=True)

    # Step 2: fusion by score summation (Algorithm 2, lines 3-4)
    fused: Dict[str, float] = {}
    for method_dict in (norm_tr, norm_yake, norm_rake):
        for kp, score in method_dict.items():
            fused[kp] = fused.get(kp, 0.0) + score

    # Step 3-4: sort descending, select top-n (Algorithm 2, lines 4-5)
    sorted_kp = sorted(fused.items(), key=lambda x: x[1], reverse=True)
    top_n = sorted_kp[:n]

    if not top_n:
        return [], time.time() - t0

    # Step 5: final re-normalisation to [0, 1]
    top_scores = [s for _, s in top_n]
    mn, mx = min(top_scores), max(top_scores)
    denom = mx - mn if mx != mn else 1.0
    final = [(kp, (s - mn) / denom) for kp, s in top_n]

    return final, time.time() - t0


h2_results, h2_time = run_hybrid2(tr_results, yake_results, rake_results, N_KEYWORDS)
print(f'Hybrid2 completed in {h2_time:.2f}s — {len(h2_results)} keyphrases extracted')

## 11. Displaying results

Helper functions display each ranked list with scores and report overlaps between methods.

In [ ]:
# Print a formatted ranking table with keyphrase and score columns
def print_table(
    title   : str,
    results : List[Tuple[str, float]],
    elapsed : float,
    note    : str = ''
) -> None:
    if not results:
        print(f'\n[{title}] No results.')
        return
    w   = max(max(len(kp) for kp, _ in results), 20)
    sep = '-' * (w + 30)
    hdr = f'  {title}' + (f'  ({note})' if note else '')
    print(f'\n+{sep}+')
    print(f'|{hdr:<{len(sep)}}|')
    print(f'|  Time:  {elapsed:.2f}s{"":<{len(sep) - 15}}|')
    print(f'+{sep}+')
    print(f'|  {"#":<5}{"Keyphrase":<{w}}  {"Score":>8}  |')
    print(f'+{sep}+')
    for i, (kp, sc) in enumerate(results, 1):
        print(f'|  {i:<5}{kp:<{w}}  {sc:>8.4f}  |')
    print(f'+{sep}+')

# Print keyphrase overlap between the four methods
def print_overlap(
    tr   : List[Tuple[str, float]],
    yake : List[Tuple[str, float]],
    rake : List[Tuple[str, float]],
    h2   : List[Tuple[str, float]],
    n    : int
) -> None:
    set_tr   = {kp.lower() for kp, _ in tr[:n]}
    set_yake = {kp.lower() for kp, _ in yake[:n]}
    set_rake = {kp.lower() for kp, _ in rake[:n]}
    set_h2   = {kp.lower() for kp, _ in h2}
    pairs = [
        ('TopicRank & YAKE',       set_tr   & set_yake),
        ('TopicRank & RAKE',       set_tr   & set_rake),
        ('YAKE      & RAKE',       set_yake & set_rake),
        ('All three',             set_tr   & set_yake & set_rake),
        ('Hybrid2   & TopicRank',  set_h2   & set_tr),
        ('Hybrid2   & YAKE',       set_h2   & set_yake),
        ('Hybrid2   & RAKE',       set_h2   & set_rake),
    ]
    sep = '-' * 72
    print(f'\n{sep}')
    print('  OVERLAP BETWEEN METHODS')
    print(sep)
    print(f'  {"Pair":<28}  {"N":>3}  Shared keyphrases')
    print(sep)
    for label, inter in pairs:
        kps = ', '.join(sorted(inter)) if inter else '-'
        if len(kps) > 36:
            kps = kps[:33] + '...'
        print(f'  {label:<28}  {len(inter):>3}  {kps}')
    print(sep)

print('Fonctions d\'affichage définies')

## 12. Results

Display rankings from all four methods, followed by an analysis of overlapping keyphrases.

In [ ]:
print_table('TOPICRANK', tr_results[:N_KEYWORDS],   tr_time,   note='higher = better')
print_table('YAKE',      yake_results[:N_KEYWORDS], yake_time, note='lower  = better')
print_table('RAKE',      rake_results[:N_KEYWORDS], rake_time, note='higher = better')
print_table('HYBRID2',   h2_results,                h2_time,   note='normalized fused scores [0,1]')
print_overlap(tr_results, yake_results, rake_results, h2_results, N_KEYWORDS)

## 13. Strengths and limitations

**Strengths**
- Computationally lightweight: no neural model or GPU is required.
- Supports English and French through YAKE, RAKE, and `pke`/spaCy.
- Combines complementary graph-based (TopicRank), positional-statistical (YAKE), and statistical-structural (RAKE) methods.
- Follows Singh & Bolla (2024), Algorithm 2, with score normalization for compatible scales.

**Limitations**
- **Unweighted score fusion:** the three methods contribute equally; the system does not learn domain-specific optimal weights.
- **Inherits the weaknesses of each method:** TopicRank depends on POS-tagging quality; RAKE is sensitive to punctuation removal; YAKE may penalize relevant terms distributed across many contexts.
- **No semantic representations:** synonymous keyphrases are treated as distinct strings.
- Min–max normalization is sensitive to outliers, which can compress the other scores.
- `clean_text()` removes punctuation before RAKE runs, removing natural phrase boundaries and potentially reducing extraction quality.